In [3]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import pickle
import xgboost as xgb

from glob import glob
import psi4
from helper_CC_ML_spacial import *

import pyscf
import pyscf.cc
import pyscf.mcscf
import ffsim
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [15]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']
basis = basis_sets[0]
n = 100

with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'out','train_names.txt'), "r") as f:
    lines = f.readlines()

ml_files = [element[:-1] for element in lines]

with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'out','test_names.txt'), "r") as f:
    lines = f.readlines()

ml_files.extend([element[:-1] for element in lines])

molecule_types = ["ammonia", "methane", "ethylene", "ethane", "water", "formaldehyde", "methanol"]
home = os.path.expanduser("~")
data_dir = os.path.join(home, "DDLUCJ", "machine_learning", "data")

# pick one file per type not in ml_files
selected = []
for mol in molecule_types:
    for fname in os.listdir(data_dir):
        if fname.startswith(mol) and fname.endswith(".xyz") and fname not in ml_files:
            selected.append(fname)
            break  # stop after first match

mol_dict = {}
for mol_file in selected:
    
    with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'data', mol_file),'r') as f:
        text=f.read()
    
    mol = psi4.geometry(text)
    print(text)
    psi4.core.clean()
    psi4.core.be_quiet()
    
    psi4.set_options({'basis': basis,
                      'scf_type':     'pk',
                      'reference':    'rohf',
                      'mp2_type':     'conv',
                      'e_convergence': 1e-8,
                      'd_convergence': 1e-8})
    
    rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
    scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
    
    A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
    mol_dict[mol_file] = (A.nmo, scf_wfn.nalpha() + scf_wfn.nbeta())


for mol_file in os.listdir(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'GDB11')):

    print(mol_file)
    if mol_file == ".ipynb_checkpoints":
        continue
    
    with open(os.path.join(os.path.expanduser('~'),'DDLUCJ', 'machine_learning', 'GDB11', mol_file),'r') as f:
        text=f.read()
    
    mol = psi4.geometry(text)
    print(text)
    psi4.core.clean()
    psi4.core.be_quiet()
    
    psi4.set_options({'basis': basis,
                      'scf_type':     'pk',
                      'reference':    'rohf',
                      'mp2_type':     'conv',
                      'e_convergence': 1e-8,
                      'd_convergence': 1e-8})
    
    rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
    scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
    
    A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=False)
    mol_dict[mol_file] = (A.nmo, scf_wfn.nalpha() + scf_wfn.nbeta())

H 1.46628 0.999842 -0.86576 
H 0.416028 0.110556 0.00208315 
H 1.75115 0.7085 0.691277 
N 1.01081 0.93139 0.0379765 
units angstrom
symmetry c1

Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 8 basis functions.
(8, 8)
(8, 8)
Building initial guess...

..initialized CCSD in 0.030 seconds.

H -0.0146124 0.692188 -0.266503 
H 1.72474 0.34345 -0.481446 
H 1.165 2.02684 -0.333225 
H 1.12196 0.941367 1.0813 
C 0.999002 0.999952 5.80823e-05 
units angstrom
symmetry c1

Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 9 basis functions.
(9, 9)
(9, 9)
Building initial guess...

..initialized CCSD in 0.039 seconds.

C -1.14137 1.73005 0.79547 
C -0.336198 0.999753 0.000194166 
H -1.30332 1.45072 1.82643 
H -1.63885 2.61532 0.417248 
H 0.164333 0.113727 0.369845 
H -0.166856 1.27314 -1.03305 
units angstrom
symmetry c1

Computing RHF reference.


...


Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 26 basis functions.
(26, 26)
(26, 26)
Building initial guess...

..initialized CCSD in 1.016 seconds.

.ipynb_checkpoints
GDB04_33.xyz
10 

O     -1.928118     -0.114744      0.178876 
C     -0.671407     -0.765721      0.264554 
C      0.284879      0.150282      0.953172 
C      1.413808      0.622382      0.408653 
H     -1.767606      0.745694     -0.244961 
H     -0.344695     -1.031447     -0.745713 
H     -0.795495     -1.687137      0.840773 
H      0.027245      0.434591      1.971228 
H      1.715326      0.362710     -0.601521 
H      2.066064      1.283390      0.971827 

Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 26 basis functions.
(26, 26)
(26, 26)
Building initial guess...

..initialized CCSD in 1.096 seconds.

GDB04_65.xyz
9 

C     -1.007357     -0.507621     -0.137822 
C      0.112792      0.215978      0.531784 
C      1.251976      0.555729     -0.072580 
F      1.491289      0.268016     -1.364635 
H     -0.791187     -0.731217     -1.187146 
H     -1.917773      0.098316     -0.099357 
H     -1.201645     -1.452609      0.378651 
H     -0.024138      0.475112      1.577742 
H      2.086041      1.078295      0.373363 

Computing RHF reference.


/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/machine_learning/injection/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})



Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 25 basis functions.
(25, 25)
(25, 25)
Building initial guess...

..initialized CCSD in 0.901 seconds.



In [16]:
mol_dict

{'ammonia157.xyz': (8, 10),
 'methane50.xyz': (9, 10),
 'ethylene42.xyz': (14, 16),
 'ethane28.xyz': (16, 18),
 'water183.xyz': (7, 10),
 'formaldehyde138.xyz': (12, 16),
 'methanol22.xyz': (14, 18),
 'GDB04_5.xyz': (21, 34),
 'GDB04_53.xyz': (26, 30),
 'GDB04_49.xyz': (26, 30),
 'GDB04_33.xyz': (26, 32),
 'GDB04_65.xyz': (25, 32)}